# 03. Лучшие майнеры на train/test: TBR и precision (вся история, без окна)

Первая половина прежних `petri_states_eval_*.ipynb` и `run_split.py`: журнал, разбиение 80/20 (seed 42), сети на train, TBR fitness на train/test и ETC precision на test.

Здесь же сети и варианты трасс test сохраняются в `out/<конфигурация>/prepared/`: `04_alignments.ipynb` работает с ними и журнал заново не строит.

Прервать и запустить снова можно: готовые метрики лежат в `align_ckpt/<модель>_tbr.json`.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "common"))
import settings as cfg

PART = "all"                         # "2013" — с окном, "all" — без окна

CONFIGS = cfg.EVAL_CONFIGS               # конфигурации с alignments


In [2]:
import time, warnings
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
import petri_pipeline as pp
import petri_states as ps
import align_parallel as ap
import pm4py

DB = pp.find_db(cfg.DB_CANDIDATES)
categorize = pp.import_categorizer(cfg.CATEGORIES_DIRS)
print("база:", DB, "| pm4py", pm4py.__version__)

tables, _ = pp.drop_full_duplicates(pp.load_tables(DB))
tables, _, _ = pp.drop_invalid_stays(tables)
hist = pp.build_history_table(tables, categorize)
obs_end = pp.observation_end(hist)
print(f"таблица истории: {len(hist):,} строк, конец наблюдения {obs_end}")



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




база: D:\diploma\jupyter lab\петри_пересчет\data\compas.db | pm4py 2.7.23.9


таблица истории: 300,249 строк, конец наблюдения 2016-04-19 07:19:36


In [3]:
rows = []
for config in CONFIGS:
    c = cfg.CONFIGS[config]
    t0 = time.time()
    log, _, _ = ps.build_state_log(hist, obs_end=obs_end, log_start=cfg.PARTS[PART]["log_start"],
                                   short_days=c["short"], long_days=c["long"], end_rule=c["end_rule"])
    for line in ps.check_state_log(log, c["end_rule"]):
        print("OK:", line)
    train, test = pp.split_people(log, cfg.TEST_SHARE, cfg.SEED)
    tr, te = pp.to_pm4py(train), pp.to_pm4py(test)
    print(f"{config}: журнал за {time.time() - t0:.0f} с; train {train.person_id.nunique():,} людей, "
          f"test {test.person_id.nunique():,}, вариантов в test {len(pm4py.get_variants(te)):,}", flush=True)
    models = ps.discover_named(tr, cfg.EVAL_MODELS[config])
    ap.save_prepared(PART, config, models, te)
    ck = cfg.out_dir(PART, config) / "align_ckpt"
    for name, m in models.items():
        t0 = time.time()
        r = ps.evaluate_split_one(name, m, tr, te, do_align=False, ckpt_dir=ck)
        rows.append(dict(config=config, **r))
        print(f"  {name}: {time.time() - t0:.0f} с, fitness test {r['fitness_tbr_test']:.3f}, "
              f"precision test {r['precision_etc_test']:.3f}", flush=True)
    del log, train, test, tr, te, models

OK: время внутри каждой трассы строго возрастает
OK: перед end: {'free': 10786, 'prison': 647, 'jail': 38, 'arrest+jail': 1, 'arrest+charge+jail': 1}
OK: перед end без заключения и не free: 0, из них не в последние 2 дня: 0
OK: free → free подряд: 0


long180_v2: журнал за 103 с; train 9,178 людей, test 2,295, вариантов в test 1,630


all long180_v2: сохранено 6 сетей и 1,630 вариантов -> D:\diploma\jupyter lab\петри_пересчет\без_окна\out\long180_v2\prepared


  DFG_p0.05: 0 с, fitness test 0.958, precision test 0.620


  DFG_p0.0: 0 с, fitness test 1.000, precision test 0.262


  HM_dep0.95: 0 с, fitness test 0.984, precision test 0.358


... (49 outputs left out)

replaying log with TBR, completed traces ::   0%|          | 0/5840 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/1626 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/30452 [00:00<?, ?it/s]

  flower: 42 с, fitness test 1.000, precision test 0.189


In [4]:
cols = ["config", "model", "places", "transitions", "silent", "train_traces", "test_traces", "fitness_tbr_train",
        "fitness_tbr_test", "pct_fit_traces_tbr_test", "precision_etc_test", "sec_tbr", "sec_prec"]
pd.DataFrame(rows)[cols].round(3)

,config,model,places,transitions,silent,train_traces,test_traces,fitness_tbr_train,fitness_tbr_test,pct_fit_traces_tbr_test,precision_etc_test,sec_tbr,sec_prec
0,long180_v2,DFG_p0.05,14,34,1,9178,2295,0.957,0.958,50.065,0.620,10.626,26.833
1,long180_v2,DFG_p0.0,14,83,1,9178,2295,1.000,1.000,100.000,0.262,17.099,75.919
2,long180_v2,HM_dep0.95,16,43,31,9178,2295,0.984,0.984,0.000,0.358,17.618,68.234
3,long180_v2,SplitMiner,21,35,23,9178,2295,0.761,0.761,7.407,0.811,35.547,13.722
4,long180_v2,IM_noise0.2,30,41,31,9178,2295,0.984,0.984,33.115,0.453,29.355,85.563
5,long180_v2,flower,3,14,2,9178,2295,1.000,1.000,100.000,0.204,7.043,34.872
6,long180_v1,DFG_p0.05,14,34,1,9178,2295,0.957,0.959,50.065,0.620,9.783,24.138
7,long180_v1,DFG_p0.0,14,82,1,9178,2295,1.000,1.000,100.000,0.263,16.226,71.856
8,long180_v1,HM_dep0.95,16,43,31,9178,2295,0.984,0.984,0.000,0.358,15.991,64.025
9,long180_v1,SplitMiner,23,34,22,9178,2295,0.733,0.733,0.000,0.877,42.365,13.212
